# 1. 환경변수 설정

In [2]:
from dotenv import load_dotenv
import os
from langchain_openai import OpenAIEmbeddings
from langchain_tavily import TavilySearch

load_dotenv(override=True)
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
GEMINI_DEFAULT_MODEL = os.getenv('GEMINI_DEFAULT_MODEL')
GPT_default_model = os.getenv('OPENAI_DEFAULT_MODEL')
embeded_model = os.getenv('EMBEDED_MODEL')
TAVILY_API_KEY =  os.getenv('TAVILY_API_KEY')
embedding_model = OpenAIEmbeddings(model=embeded_model)

# 2. 잼민이 사용해서 요약정리 (pydantic으로 구조 정형화)

In [3]:
from google import genai
from typing import Literal
from pydantic import BaseModel

Character = Literal["치이카와", "하치와레", "우사기", "모몽가", "쿠리만주", "랏코", "시사", "기타"]

class EpisodeSummary(BaseModel):
    characters: list[Character]
    summary: str
    key_scenes: list[str]
    objects: list[str]

PROMPT = """이 치이카와 에피소드를 한국어로 정리해줘.

캐릭터는 반드시 아래 이름으로 적어. 동물 이름(토끼, 고양이 등)이나 생김새(가르마 등)로 쓰지 마.
- 치이카와: 하얀 몸, 작고 겁이 많은 주인공
- 하치와레: 흰 고양이 같은 캐릭터, 머리 위쪽이 파란빛 회색으로 양쪽으로 갈라진 무늬
- 우사기: 노란 토끼, "우라!", "하아?" 같은 소리를 냄
- 모몽가: 하날다람쥐. 하얀 몸에 크고 동그란 눈, 팔과 몸 사이에 비막. 관심과 칭찬을 요구하는 제멋대로인 성격
- 쿠리만주: 밤만쥬처럼 생긴 갈색 캐릭터. 술과 안주를 좋아하고 마신 뒤 "하~" 하고 한숨을 쉼
- 랏코: 해달. 회갈색 몸, 토벌 실력이 뛰어난 상위 랭커
- 시사: 오키나와 시사 같은 캐릭터. 주황빛 갈기, 라멘 가게 수습생
- 목록에 없는 캐릭터는 "기타"
summary와 key_scenes에서도 같은 이름을 써.

- characters: 등장 캐릭터
- summary: 줄거리 3~5문장 (장면을 구체적으로)
- key_scenes: 인상적인 장면 묘사 (표정, 행동, 감정 포함)
- objects: 등장하는 음식/물건/장소"""

client = genai.Client() 

def summarize(url):
    res = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=[{"file_data": {"file_uri": url}}, PROMPT],
        config={
            "response_mime_type": "application/json",
            "response_schema": EpisodeSummary,
            "temperature": 0,       
        },
    )
    return res.text

# print(summarize("https://www.youtube.com/watch?v=kJnFGdmFqTg"))

# 3. Chroma로 db 저장

In [3]:
import json, time
from pathlib import Path
import pandas as pd
from langchain_chroma import Chroma
from langchain_core.documents import Document

CSV_PATH = "episodes.csv"
RAW_DIR = Path("raw_json")
RAW_DIR.mkdir(exist_ok=True)

# ---------- 헬퍼 ----------
def to_text(x) -> str:
    if isinstance(x, str):
        return x
    if isinstance(x, list):
        return " ".join(to_text(i) for i in x)
    if isinstance(x, dict):
        return " / ".join(f"{k}: {to_text(v)}" for k, v in x.items())
    return str(x)

def load_episodes(path: str) -> pd.DataFrame:
    for enc in ("utf-8-sig", "cp949"):
        try:
            df = pd.read_csv(path, encoding=enc, dtype=str)
            break
        except UnicodeDecodeError:
            continue
    df = df.dropna(subset=["episode_id", "video_url"])
    df = df.drop_duplicates(subset=["episode_id"])
    return df.apply(lambda col: col.str.strip())

def to_documents(episode_id: str, raw: str):
    data = json.loads(raw)

    metadata = {
        "characters": ", ".join(dict.fromkeys(map(str, data.get("characters", [])))),
        "objects": ", ".join(map(str, data.get("objects", []))),
    }

    docs = [Document(page_content=to_text(data["summary"]), metadata=metadata)]
    ids = [f"{episode_id}-summary"]

    for i, scene in enumerate(data.get("key_scenes", [])):
        docs.append(Document(page_content=to_text(scene), metadata=metadata))
        ids.append(f"{episode_id}-scene-{i}")

    return docs, ids

# ---------- 1단계: CSV의 url로 Gemini 요약 ----------
df = load_episodes(CSV_PATH) 
# df = df.head(5)          # 테스트용. 확인 후 삭제
# print(len(df), "개 에피소드")


failed = []
for row in df.itertuples(index=False):
    out = RAW_DIR / f"{row.episode_id}.json"
    if out.exists():
        continue
    for attempt in range(3):
        try:
            raw = summarize(row.video_url)
            json.loads(raw)
            out.write_text(raw, encoding="utf-8")   # Gemini 결과 그대로 저장
            print(f"[OK] {row.episode_id}")
            break
        except Exception as e:
            wait = 2 ** attempt * 5
            print(f"[RETRY {attempt+1}] {row.episode_id}: {e} → {wait}s 대기")
            time.sleep(wait)
    else:
        failed.append(row.episode_id)
    time.sleep(2)

print("실패:", failed)

# ---------- 2단계: Chroma 적재 ----------
vectorstore = Chroma(
    collection_name="chiikawa",
    embedding_function=embedding_model,
    persist_directory="chroma3",
    collection_metadata={"hnsw:space": "cosine"},
)

for path in sorted(RAW_DIR.glob("*.json")):
    episode_id = path.stem                      # 파일 이름 = ep001
    docs, ids = to_documents(episode_id, path.read_text(encoding="utf-8"))
    vectorstore.add_documents(docs, ids=ids)
    print(f"[적재] {episode_id}: {len(docs)}건")

[OK] ep001
[OK] ep002
[OK] ep003
[OK] ep004
[OK] ep005
[OK] ep006
[OK] ep007
[OK] ep008
[OK] ep009
[OK] ep010
[OK] ep011
[OK] ep012
[OK] ep013
[OK] ep014
[OK] ep015
[OK] ep016
[OK] ep017
[OK] ep018
[OK] ep019
[OK] ep020
[OK] ep021
[OK] ep022
[OK] ep023
[OK] ep024
[OK] ep025
[OK] ep026
[OK] ep027
[OK] ep028
[OK] ep029
[OK] ep030
[OK] ep031
[OK] ep032
[OK] ep033
[OK] ep034
[OK] ep035
[OK] ep036
[OK] ep037
[OK] ep038
[OK] ep039
[OK] ep040
[OK] ep041
[OK] ep042
[OK] ep043
[OK] ep044
[OK] ep045
[OK] ep046
[OK] ep047
[OK] ep048
[OK] ep049
[OK] ep050
[OK] ep051
[OK] ep052
[OK] ep053
[OK] ep054
[OK] ep055
[OK] ep056
[OK] ep057
[OK] ep058
[OK] ep059
[OK] ep060
[OK] ep061
[OK] ep062
[OK] ep063
[OK] ep064
[OK] ep065
[OK] ep066
[OK] ep067
[OK] ep068
[OK] ep069
[OK] ep070
[OK] ep071
[OK] ep072
[OK] ep073
[OK] ep074
[OK] ep075
[OK] ep076
[OK] ep077
[OK] ep078
[OK] ep079
[OK] ep080
[OK] ep081
[OK] ep082
[OK] ep083
[OK] ep084
[OK] ep085
[OK] ep086
[OK] ep087
[OK] ep088
[OK] ep089
[OK] ep090
[OK] ep091

## 데이터 확인용

In [4]:
print(vectorstore._client.get_settings().persist_directory)   # 어느 폴더인지
print(vectorstore._collection.count())                         # 몇 건인지
print(vectorstore.get()["ids"][:10])                           # id 모양

NameError: name 'vectorstore' is not defined

In [5]:
r = vectorstore.get(include=["metadatas"])

for id_, meta in zip(r["ids"], r["metadatas"]):
    if id_.endswith("-summary"):          # 에피소드당 한 번만 출력
        print(id_)
        print("  characters:", meta["characters"])
        print("  objects   :", meta["objects"])

print(vectorstore.get(ids=["ep001-summary"])["documents"][0])

ep001-summary
  characters: 치이카와, 하치와레, 우사기
  objects   : 푸딩, 핫케이크, 뒤집개, 버터, 접시, 들판
ep002-summary
  characters: 치이카와, 하치와레, 우사기
  objects   : 책, 이집트 피라미드 책, 스핑크스, 들판
ep003-summary
  characters: 치이카와, 우사기
  objects   : 오징어 간식, 멜론 소다, 컵케이크 모양 스프 구덩이
ep004-summary
  characters: 치이카와, 우사기
  objects   : 브로콜리, 소스통, 포크, 접시, 피자 만두
ep005-summary
  characters: 치이카와, 하치와레, 우사기
  objects   : 크레용, 종이, 까르메라우 라면, 냄비, 그릇, 젓가락, 꽃 화관, 초원
ep006-summary
  characters: 치이카와, 하치와레, 우사기
  objects   : 팬케이크, 완두콩 앙금, 풀밭, 나무, 산
ep007-summary
  characters: 치이카와, 하치와레, 우사기
  objects   : 스마트폰, 요구르트, 자판기, 카드
ep008-summary
  characters: 치이카와, 하치와레, 우사기
  objects   : 야쿠르트, 나무 그루터기, 숲
ep009-summary
  characters: 치이카와, 우사기
  objects   : 코코아, 과자, 종이 냅킨, 테이블, 모래시계, 돼지 컵라면, 포크
ep010-summary
  characters: 치이카와, 하치와레, 우사기
  objects   : 녹색 간식 상자, 꼬치, 잔디밭
ep011-summary
  characters: 하치와레, 치이카와, 우사기
  objects   : 기타, 통나무, 풀숲, 스크린, 유모차
ep012-summary
  characters: 치이카와, 하치와레, 우사기
  objects   : 초밥, 초밥 도시락, 주전자, 오차즈케 그릇, 찻잔, 골동품 상자,

# 4. Chroma local DB에서 읽어오기

In [6]:
from langchain_chroma import Chroma

vectorstore = Chroma(
    collection_name="chiikawa",          # 저장할 때와 같은 이름
    embedding_function=embedding_model,  # 저장할 때와 같은 임베딩 모델
    persist_directory="chroma3",
)

# 5. retriever 생성

In [7]:
# 2. retriever 생성
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 10},
)

# 6. LLM 생성 및 질문 체인

In [8]:
# 3. LLM 생성 및 질문 체인

from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

retriever = vectorstore.as_retriever(search_type="mmr",
                                     search_kwargs={"k": 3, "fetch_k": 10})

llm = init_chat_model(f"openai:{GPT_default_model}")

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "너는 치이카와 안내자야. 아래 [자료]만 근거로 한국어로 답해. "
     "자료에 없는 내용은 모른다고 말하고, 답 끝에 참고한 에피소드 아이디를 적어줘.\n\n"
     "[자료]\n{context}"),
    ("human", "{question}"),
])

def format_docs(docs):
    blocks = []
    for d in docs:
        doc_id = d.id or ""                         
        episode_id = doc_id.split("-")[0]          
        kind = "줄거리" if doc_id.endswith("-summary") else "장면"
        blocks.append(
            f"[{episode_id} / {kind}] (등장: {d.metadata.get('characters', '')})\n"
            f"{d.page_content}"
        )
    return "\n\n".join(blocks)

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 7 DB에 없을시 웹 검색 도구

In [10]:
# 4. 웹 검색 fallback
from langchain_tavily import TavilySearch
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

web_search = TavilySearch(max_results=3)
THRESHOLD = 0.3                                    # ← 0.4 → 0.3

# ↓ 추가: 자료로 답할 수 있는지 LLM이 판단
grade_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "아래 [자료]만으로 [질문]에 답할 수 있으면 yes, 없으면 no라고만 답해.\n\n[자료]\n{context}"),
    ("human", "[질문] {question}"),
])
grader = grade_prompt | llm | StrOutputParser()

# ↓ 변경: 점수로 1차 거르고, 남은 건 LLM이 2차 판단
def search_chroma(question):
    results = vectorstore.similarity_search_with_relevance_scores(question, k=3)
    docs = [d for d, score in results if score >= THRESHOLD]
    if not docs:
        return []
    verdict = grader.invoke({"context": format_docs(docs), "question": question})
    return docs if verdict.strip().lower().startswith("yes") else []

def search_web(question):
    query = question if "치이카와" in question else f"치이카와 {question}"   # ← 중복 방지
    res = web_search.invoke({"query": query})
    return "\n\n".join(f"[웹] {r['title']} ({r['url']})\n{r['content']}"
                       for r in res["results"])

answer_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "너는 치이카와 안내자야. 아래 [자료]만 근거로 한국어로 답해. "
     "자료에 없는 내용은 모른다고 말해. 답 끝에 출처(에피소드 아이디 또는 웹 URL)를 적어줘.\n\n"
     "[자료]\n{context}"),
    ("human", "{question}"),
])

def ask(question):
    docs = search_chroma(question)
    if docs:
        context, source = format_docs(docs), "DB"
    else:
        context, source = search_web(question), "웹"

    answer = (answer_prompt | llm | StrOutputParser()).invoke(
        {"context": context, "question": question})
    print(f"[출처: {source}]\n{answer}")

In [11]:
questions = [
    "치이카와가 라면 먹는 에피소드 알려줘",   # DB로 가야 할 질문
    "치이카와가 피자 만두 먹는 장면",
    "치이카와 작가가 누구야?",               # 웹으로 가야 할 질문
    "오늘 서울 날씨 어때?",
]

for q in questions:
    print(q)
    for d, score in vectorstore.similarity_search_with_relevance_scores(q, k=3):
        print(f"  {score:.3f}  {d.id}  {d.page_content[:40]}")
    print()

치이카와가 라면 먹는 에피소드 알려줘
  0.636  ep242-scene-2  치이카와와 하치와레가 과일 탄산음료를 들고 오뎅과 무를 반으로 나눠 먹으
  0.627  ep106-scene-2  치이카와가 큰 가방에서 간식을 꺼내 하치와레와 함께 기뻐하는 장면
  0.626  ep165-scene-1  치이카와와 하치와레가 시사에게 다가와 간식을 받고 기뻐하는 장면

치이카와가 피자 만두 먹는 장면
  0.712  ep106-scene-2  치이카와가 큰 가방에서 간식을 꺼내 하치와레와 함께 기뻐하는 장면
  0.709  ep242-scene-2  치이카와와 하치와레가 과일 탄산음료를 들고 오뎅과 무를 반으로 나눠 먹으
  0.709  ep025-scene-1  치이카와가 화장실에 다녀오다 간식을 떨어뜨리고 울먹이는 표정을 짓는 장면

치이카와 작가가 누구야?
  0.584  ep098-scene-1  치이카와가 두려움에 떨며 눈물을 글썽이는 장면
  0.558  ep074-scene-2  치이카와와 하치와레가 손을 맞잡고 기뻐하는 장면
  0.558  ep151-scene-2  치이카와가 둥근 탁자에 앉아 연필로 공책에 그림을 그리며 고민하는 모습.

오늘 서울 날씨 어때?
  0.317  ep195-scene-2  세 친구가 나란히 하늘을 올려다보며 오데를 떠올리는 장면
  0.315  ep173-scene-0  치이카와, 하치와레, 우사기가 요정 날개를 달고 하늘을 날며 기뻐하는 장
  0.308  ep218-scene-1  아침에 알람시계를 들고 일어난 치이카와가 들판을 신나게 뛰어다닌다.



In [14]:
for word in ["라면", "피자"]:
    r = vectorstore.get(where_document={"$contains": word})
    print(word, len(r["ids"]), "건")
    for id_, doc in zip(r["ids"][:5], r["documents"][:5]):
        print("  ", id_, "|", doc[:50])

라면 14 건
   ep005-summary | 치이카와가 방 안에서 그림을 그리는 동안, 하치와레가 '까르메라우' 봉지를 들고 들어와 자
   ep005-scene-1 | 하치와레가 '까르메라우' 라면 봉지를 신나게 들어 보이며 치이카와에게 보여주는 장면
   ep005-scene-2 | 치이카와와 하치와레가 마주 앉아 맛있게 라면을 후루룩 먹고 행복해하는 장면
   ep009-summary | 치이카와가 야외 테이블에서 코코아와 과자를 먹으며 여유로운 시간을 보낸다. 이때 우사기가 
   ep009-scene-2 | 치이카와와 우사기가 나란히 앉아 김이 모락모락 나는 컵라면을 맛있게 흡입하는 장면
피자 10 건
   ep004-summary | 치이카와와 우사기가 야외 테이블에서 브로콜리를 먹기 시작합니다. 우사기는 브로콜리에 소스를
   ep004-scene-2 | 우사기가 거대한 피자 만두를 한 입 베어 물고 치즈가 길게 늘어나는 모습에 당황해 울먹이는
   ep115-summary | 하늘에서 떨어진 피자를 발견한 하치와레와 다른 친구들이 피자를 차지하려고 합니다. 우사기가
   ep115-scene-0 | 하치와레와 다른 캐릭터가 하늘에 떠 있는 거대한 피자를 신기하게 바라보며 피자일지 복감치야
   ep115-scene-1 | 우사기가 피자를 잡기 위해 높게 점프하지만 손이 닿지 않아 아쉬워하는 장면


In [12]:
ask("치이카와가 라면 먹는 에피소드 알려줘")   # [출처: DB]
ask("치이카와 작가가 누구야?")               # [출처: 웹]
ask("오늘 서울 날씨 어때?")                   # [출처: 웹]

[출처: 웹]
치이카와가 라면을 먹는 장면은 자료 기준으로 다음과 같아요.

- **‘챠리메라 라면’ 에피소드**: 하치와레가 라면 이름을 ‘챠리메라’라고 잘못 읽어요. 치이카와가 작게 바로잡아 주려 하지만 하치와레가 듣지 못하자 함께 먹고, 맛있게 먹은 뒤 둘이 손을 들고 “최고~”라고 외쳐요.
- **‘챠루메라 라면’ 에피소드**: 하치와레가 전에 먹은 라면의 이름이 사실 ‘챠루메라’였다고 부끄러워하자, 치이카와가 웃어 줘요. 그리고 다시 함께 “챠루메라 최고~”라고 해요.
- **‘부타멘 컵라면’ 에피소드**: 치이카와는 모래시계를 보며 컵라면을 기다려요. 우사기가 먼저 먹기 시작하자 망설이다가 함께 먹어요.
- **‘기운이 났다’ 에피소드**: 하치와레가 몬스터를 쫓아낸 뒤 울던 치이카와가, 이후 하치와레와 라멘을 먹으며 기운을 차려요.

출처: [치이카와(먼작귀) - 나무위키](https://namu.wiki/w/%EC%B9%98%EC%9D%B4%EC%B9%B4%EC%99%80(%EB%A8%BC%EC%9E%91%EA%B7%80))
[출처: 웹]
치이카와의 작가는 **나가노(ナガノ)**입니다. 출처: [위키백과](https://ko.wikipedia.org/wiki/%EB%A8%BC%EA%B0%80_%EC%9E%91%EA%B3%A0_%EA%B7%80%EC%97%AC%EC%9A%B4_%EB%85%80%EC%84%9D)
[출처: 웹]
치이카와 안내자가 확인했어요! 오늘 **2026년 10월 21일** 서울은 **구름이 많고 한때 비**가 올 전망이고, 기온은 **12~15℃**예요. 미세먼지와 초미세먼지는 모두 **좋음**으로 예보돼 있어요. ☁️🌧️

※ 제공된 날씨 자료의 날짜 표기가 서로 일치하지 않아, 10월 21일로 적힌 서울 공식 관광정보 예보를 기준으로 안내했어요.

출처: https://korean.visitseoul.net/weather


In [9]:
for d in retriever.invoke("빵 먹는 장면"):
    print(d.id)

ep014-scene-1
ep016-scene-1
ep009-scene-2


In [13]:
print(chain.invoke("우사기가 뭐고 어떻게 생겼어?"))

자료에서 우사기는 치이카와·하치와레와 함께 등장하는 인물이에요. 망치를 들고 시계를 때리거나, 작은 생명체와 뛰놀고, 강에서 물고기를 잡는 모습이 나와요. 하지만 생김새는 자료에 설명되어 있지 않아 알 수 없어요.

참고 에피소드: ep225, ep088, ep160
